In [24]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
import re

In [20]:
def _safe_llm_call( prompt, model="Qwen/Qwen2.5-7B-Instruct"):
        """Calls LLM and ensures the output is a clean Python Dictionary."""
        tokenizer = AutoTokenizer.from_pretrained(model, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(
            model, device_map="auto", torch_dtype=torch.bfloat16, trust_remote_code=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=1024, temperature=0.1)
        resp = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        
        # Robust JSON extraction using Regex to remove markdown junk
        json_match = re.search(r'\{.*\}', resp, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group())
            except:
                return {"error": "parsing_failed", "content": resp}
        return {"error": "no_json_found"}

In [21]:
text = """Malik was married in 2002 to Ayesha Siddiqui from Hyderabad, India which ended in divorce on 7 April 2010.[9][10]

Malik reportedly dated Miss India-turned-actress Sayali Bhagat, whom he met in 2008 when he was supposed to make his Bollywood debut, but they eventually separated when he announced his engagement to Indian tennis player Sania Mirza.[11] Writing in 2008 for CricInfo, Andrew McGlashan called Bhagat his "real-life girlfriend."[12] Both Malik and Bhagat, however, denied such reports, calling them publicity stunts to market their under-production film.[13][14]

On 12 April 2010 he married Sania Mirza in a traditional Hyderabadi Muslim wedding ceremony at the Taj Krishna Hotel in Hyderabad, India, followed by Pakistani wedding customs.[15][16][17][18] Their walima ceremony was held in Shoaib's native Sialkot, Pakistan.[19] Their wedding received media and online attention across the world.[20] The couple announced their pregnancy via social media on 23 April 2018.[21][22] Their first child, a boy, was born on 30 October 2018.[23] In January of 2024 Mirza's family announced that she had gotten a divorce from Malik a few months prior.[24]

On 19 January 2024, Shoaib married Pakistani TV actress Sana Javed in a private Nikah ceremony at her home in Karachi."""

In [29]:
url = "https://en.wikipedia.org/wiki/Shoaib_Malik"

In [30]:
prompt = f"""<|im_start|>system
Take the follwoing url and read its text, and as a smart notes, generate summary, extract 3 topics that are most relevant, write down most important actions items,
Return JSON: {{"summary": "str", "topics": ["...", "..."], "action_items": ["...","..."]}} 
URL = '''{url}'''
<|im_end|>
<|im_assistant|>"""
        


In [31]:
res = _safe_llm_call(prompt)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [32]:
res

{'summary': 'Shoaib Malik is a former Pakistani cricketer who played for Pakistan in Test, One Day International (ODI), and Twenty20 International (T20I) formats. He was known for his batting skills and leadership qualities. Malik captained the Pakistan national cricket team from 2011 to 2016 and was part of the team that won the 2011 ICC Cricket World Cup. He retired from international cricket in 2017 after playing 154 ODI matches and 48 Test matches.',
 'topics': ['Cricket Career',
  'Leadership',
  'International Cricket Achievements'],
 'action_items': ["Research Malik's batting techniques and strategies used during his career.",
  "Explore the impact of Malik's leadership on the Pakistan national cricket team.",
  "Analyze Malik's role in the 2011 ICC Cricket World Cup victory."]}